In [ ]:
import os
os.environ['PYTHONHASHSEED'] = '0'
import warnings
warnings.filterwarnings('ignore')

import json as _json
import math
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, fbeta_score, accuracy_score,
)

seed = 42
np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FEATURE_COLS   = ['rv_centered', 'rv_err', 'RHKp', 'Halpha']  # 4 raw channels, no positional encoding
N_INPUT        = len(FEATURE_COLS)                              # 4
MAX_SEQ_LEN    = 100           # truncate long-cadence stars (matches cnn_stripped)
N_EPOCHS       = 100           # fixed epochs per fold (no early stopping)
BATCH_SIZE     = 32
LR             = 1e-3
WEIGHT_DECAY   = 5e-3
DROPOUT        = 0.3
HIDDEN         = 64
TAIL           = 32
HEAD_HIDDEN    = 32
WARMUP_EPOCHS  = 5
GRAD_CLIP      = 1.0

N_REPS   = 5
N_FOLDS  = 5

OBS_PKL = '/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl'

observations = pd.read_pickle(OBS_PKL)
print(f'Observations: {len(observations)} rows, {observations["star_name"].nunique()} stars')
print(f'Device: {device}')
print(f'Features: {FEATURE_COLS}')
print(f'Config: epochs={N_EPOCHS}, batch={BATCH_SIZE}, lr={LR}, wd={WEIGHT_DECAY}, dropout={DROPOUT}')

In [ ]:
grouped  = observations.groupby('star_name', sort=True)
stars    = list(grouped.groups.keys())
labels   = np.array([int(grouped.get_group(s)['has_exoplanets'].iloc[0]) for s in stars],
                    dtype=int)
n_stars  = len(stars)
print(f'n_stars = {n_stars}, positives = {int(labels.sum())}, negatives = {int((labels==0).sum())}')
print(f'Class ratio 1:{n_stars / max(labels.sum(), 1):.1f}')

In [ ]:
star_groups = {s: grouped.get_group(s).sort_values('bjd') for s in stars}

def build_sequence(star):
    g = star_groups[star]
    return g[FEATURE_COLS].values.astype(np.float32)

print(f'Building sequences for {n_stars} stars...')
star_seqs = {s: build_sequence(s) for s in stars}
seq_lens  = [len(star_seqs[s]) for s in stars]
print(f'Done. Sequence lengths: min={min(seq_lens)}, max={max(seq_lens)}, median={int(np.median(seq_lens))}')
print(f'First star: {stars[0]}, shape={star_seqs[stars[0]].shape}')

In [ ]:
class MaskedConv1dBlock(nn.Module):
    """Conv1d('same') + BatchNorm + GELU + Dropout."""
    def __init__(self, in_ch, out_ch, kernel_size, dropout=0.3):
        super().__init__()
        pad = kernel_size // 2  # 'same' padding (kernel is odd)
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=pad)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.act(self.bn(self.conv(x))))

class MaskedAvgMaxPool(nn.Module):
    """Concatenates masked global mean and masked global max pooling."""
    def forward(self, x, mask):
        m = mask.unsqueeze(1).type_as(x)  # (B, 1, T)
        denom = m.sum(dim=2).clamp_min(1.0)
        avg = (x * m).sum(dim=2) / denom          # (B, C)
        masked_x = x.masked_fill(m == 0, float('-inf'))
        mx = masked_x.max(dim=2).values           # (B, C)
        mx = torch.nan_to_num(mx, nan=0.0, posinf=0.0, neginf=0.0)
        return torch.cat([avg, mx], dim=1)         # (B, 2C)

class StrippedCNN(nn.Module):
    """3-layer 1D CNN on raw per-observation features, no positional encoding."""
    def __init__(self, in_dim=4, hidden=64, tail=32, head_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = MaskedConv1dBlock(in_dim,  hidden, 5, dropout)
        self.conv2 = MaskedConv1dBlock(hidden, hidden, 5, dropout)
        self.conv3 = MaskedConv1dBlock(hidden, tail,    3, dropout)
        self.pool  = MaskedAvgMaxPool()  # concat → (B, 2*tail = 64)
        self.head = nn.Sequential(
            nn.Linear(2 * tail, head_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, x, mask):
        x = x.transpose(1, 2)      # (B, T, C) → (B, C, T)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.pool(x, mask)
        return self.head(x).squeeze(-1)  # logits (B,)

dummy = StrippedCNN(in_dim=N_INPUT, hidden=HIDDEN, tail=TAIL,
                    head_hidden=HEAD_HIDDEN, dropout=DROPOUT)
n_params = sum(p.numel() for p in dummy.parameters() if p.requires_grad)
print(f'StrippedCNN parameters: {n_params:,}')
x0 = torch.randn(4, 50, N_INPUT)
m0 = torch.ones(4, 50, dtype=torch.bool)
with torch.no_grad():
    y0 = dummy(x0, m0)
print(f'Smoke forward: in={tuple(x0.shape)}, mask={tuple(m0.shape)}, out={tuple(y0.shape)})')
del dummy, x0, m0, y0

In [ ]:
class StarDataset(Dataset):
    def __init__(self, sequences, labels):
        self.data = list(zip(sequences, labels))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq, lab = self.data[idx]
        return torch.tensor(seq), torch.tensor(lab, dtype=torch.float32)

def collate_fn(batch):
    """Pad variable-length sequences to longest in batch, return (padded, mask, labels)."""
    sequences, labels = zip(*batch)
    lengths = [s.shape[0] for s in sequences]
    max_len = max(lengths)
    padded = torch.zeros(len(sequences), max_len, N_INPUT)
    mask   = torch.zeros(len(sequences), max_len, dtype=torch.bool)
    for i, (seq, length) in enumerate(zip(sequences, lengths)):
        padded[i, :length] = seq
        mask[i, :length] = True
    labels = torch.stack(labels)
    return padded, mask, labels


In [ ]:
def train_one_fold(train_idx, test_idx, rep_seed, verbose=False):
    """Train a fresh StrippedCNN on train_idx; return probabilities on test_idx."""
    torch.manual_seed(rep_seed); torch.cuda.manual_seed_all(rep_seed)
    np.random.seed(rep_seed)

    train_seqs_raw = [star_seqs[stars[i]] for i in train_idx]
    test_seqs_raw  = [star_seqs[stars[i]] for i in test_idx]
    y_train_fold = labels[train_idx].astype(np.float32).tolist()
    y_test_fold  = labels[test_idx].astype(np.float32).tolist()

    train_all = np.concatenate(train_seqs_raw, axis=0)  # (n_train_obs, 4)
    feat_mean = train_all.mean(axis=0, keepdims=True)
    feat_std  = np.clip(train_all.std(axis=0, keepdims=True), 1e-8, None)

    def standardize(seq_list):
        return [(s.astype(np.float32) - feat_mean) / feat_std for s in seq_list]

    train_seqs = standardize(train_seqs_raw)
    test_seqs  = standardize(test_seqs_raw)

    for seq_list in [train_seqs, test_seqs]:
        for i in range(len(seq_list)):
            if len(seq_list[i]) > MAX_SEQ_LEN:
                seq_list[i] = seq_list[i][:MAX_SEQ_LEN]

    train_ds = StarDataset(train_seqs, y_train_fold)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, drop_last=False)

    model = StrippedCNN(in_dim=N_INPUT, hidden=HIDDEN, tail=TAIL,
                        head_hidden=HEAD_HIDDEN, dropout=DROPOUT).to(device)

    n_pos = int(sum(y_train_fold))
    n_neg = int(len(y_train_fold) - n_pos)
    pos_w = torch.tensor([n_neg / n_pos]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / WARMUP_EPOCHS
        progress = (epoch - WARMUP_EPOCHS) / max(1, N_EPOCHS - WARMUP_EPOCHS)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = LambdaLR(optimizer, lr_lambda)

    model.train()
    for ep in range(N_EPOCHS):
        for padded, mask, yb in train_loader:
            padded, mask, yb = padded.to(device), mask.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(padded, mask), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()
        scheduler.step()
        if verbose and (ep + 1) % 10 == 0:
            print(f'    ep {ep+1:3d}: loss={loss.item():.4f}')

    model.eval()
    test_ds = StarDataset(test_seqs, y_test_fold)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                             collate_fn=collate_fn, drop_last=False)
    all_logits = []
    with torch.no_grad():
        for padded, mask, _ in test_loader:
            padded, mask = padded.to(device), mask.to(device)
            all_logits.append(model(padded, mask).cpu().numpy())
    logits = np.concatenate(all_logits)

    logits = np.nan_to_num(logits, nan=0.0, posinf=35.0, neginf=-35.0)
    probs = 1.0 / (1.0 + np.exp(-logits))
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0).astype(np.float32)
    return probs


In [ ]:
all_oof_probs = np.zeros((n_stars, N_REPS), dtype=np.float32)
all_oof_preds = np.zeros((n_stars, N_REPS), dtype=int)
rep_metrics   = {'rep': [], 'pr_auc': [], 'roc_auc': [],
                 'f1': [], 'f05': [], 'precision': [], 'recall': []}

for rep in range(N_REPS):
    oof_preds = np.zeros(n_stars, dtype=int)
    oof_probs = np.zeros(n_stars, dtype=np.float32)
    rep_seed  = 42 + rep
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=rep_seed)
    fold_times = []
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(np.zeros(n_stars), labels)):
        t0 = time.time()

        inner_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=rep_seed)
        inner_probs_fold = np.zeros(len(train_idx), dtype=np.float32)
        for i_train, i_val in inner_skf.split(np.zeros(len(train_idx)), labels[train_idx]):
            inner_probs_fold[i_val] = train_one_fold(train_idx[i_train], train_idx[i_val], rep_seed)

        vp_in, vr_in, vt_in = precision_recall_curve(labels[train_idx], inner_probs_fold)
        vf1_in = 2 * vp_in * vr_in / (vp_in + vr_in + 1e-8)
        fold_thr = float(vt_in[int(np.argmax(vf1_in))]) if len(vt_in) > 0 else 0.5

        probs = train_one_fold(train_idx, test_idx, rep_seed, verbose=False)
        oof_probs[test_idx] = probs
        oof_preds[test_idx] = (probs >= fold_thr).astype(int)
        fold_times.append(time.time() - t0)
        if (fold_idx + 1) % 2 == 0:
            print(f'  rep {rep} fold {fold_idx+1:2d}/{N_FOLDS} done ({np.mean(fold_times):.0f}s/fold avg)')
        del probs
    all_oof_probs[:, rep] = oof_probs
    all_oof_preds[:, rep] = oof_preds

    roc = roc_auc_score(labels, oof_probs)
    pr  = average_precision_score(labels, oof_probs)
    cm = confusion_matrix(labels, oof_preds)
    tn, fp, fn, tp = cm.ravel()
    prc = tp / (tp+fp) if (tp+fp) > 0 else 0.0
    rec = tp / (tp+fn) if (tp+fn) > 0 else 0.0
    f1  = f1_score(labels, oof_preds, zero_division=0)
    f05 = fbeta_score(labels, oof_preds, beta=0.5, zero_division=0)
    rep_metrics['rep'].append(rep); rep_metrics['pr_auc'].append(pr)
    rep_metrics['roc_auc'].append(roc); rep_metrics['f1'].append(f1)
    rep_metrics['f05'].append(f05)
    rep_metrics['precision'].append(prc); rep_metrics['recall'].append(rec)
    print(f'  rep {rep} (seed {rep_seed}): pr_auc={pr:.4f} roc_auc={roc:.4f} f1={f1:.4f} f0.5={f05:.4f} p={prc:.3f} r={rec:.3f}')

print(f"\nOOF evaluation complete: {N_FOLDS * N_REPS} fits total.")

In [ ]:
rep_df = pd.DataFrame(rep_metrics)

print("\nper-rep pr_auc:")
for _, row in rep_df.iterrows():
    print(f"  rep {int(row['rep'])}: pr_auc={row['pr_auc']:.4f}")

print(f"\naggregate (n={N_REPS} reps):")
for m in ['pr_auc', 'roc_auc', 'f1', 'f05', 'precision', 'recall']:
    v = rep_df[m].values
    print(f"  {m}: {v.mean():.4f} +/- {v.std(ddof=1):.4f} (min={v.min():.4f}, max={v.max():.4f})")

avg_oof = all_oof_probs.mean(axis=1)
combined_pr  = average_precision_score(labels, avg_oof)
combined_roc = roc_auc_score(labels, avg_oof)
combined_preds = (all_oof_preds.mean(axis=1) >= 0.5).astype(int)
combined_f1   = f1_score(labels, combined_preds, zero_division=0)
combined_f05  = fbeta_score(labels, combined_preds, beta=0.5, zero_division=0)
cm = confusion_matrix(labels, combined_preds)
tn, fp, fn, tp = cm.ravel()
combined_p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
combined_r = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print(f"\ncombined oof (avg across {N_REPS} reps):")
print(f"  pr_auc: {combined_pr:.4f}")
print(f"  roc_auc: {combined_roc:.4f}")
print(f"  f1: {combined_f1:.4f}  f0.5: {combined_f05:.4f}")
print(f"  p={combined_p:.3f}  r={combined_r:.3f} (TN={tn} FP={fp} FN={fn} TP={tp})")

In [ ]:
def bootstrap_roc_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for ROC-AUC via the percentile method."""
    from sklearn.metrics import roc_auc_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap ROC-AUC")

    point = float(roc_auc_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aucs[i] = point  # fall back to point estimate if degenerate
            continue
        aucs[i] = roc_auc_score(yt, ys)
    lo = float(np.percentile(aucs, 2.5))
    hi = float(np.percentile(aucs, 97.5))
    return point, lo, hi

def bootstrap_pr_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for PR-AUC (average precision) via the percentile method."""
    from sklearn.metrics import average_precision_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap PR-AUC")

    point = float(average_precision_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aps = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aps[i] = point  # fall back to point estimate if degenerate
            continue
        aps[i] = average_precision_score(yt, ys)
    lo = float(np.percentile(aps, 2.5))
    hi = float(np.percentile(aps, 97.5))
    return point, lo, hi
pr_point, pr_lo, pr_hi = bootstrap_pr_auc(labels, avg_oof)
roc_point, roc_lo, roc_hi = bootstrap_roc_auc(labels, avg_oof)
print("bootstrap 95% ci (200 resamples):")
print(f"  pr_auc: {pr_point:.4f} [{pr_lo:.4f}, {pr_hi:.4f}]")
print(f"  roc_auc: {roc_point:.4f} [{roc_lo:.4f}, {roc_hi:.4f}]")